# OCR Extraction Pipeline — Setup & Test

Full litmus test for the extraction pipeline.

**Before starting:** Upload your sample PDFs/TIFFs to `/ocr/` using
Jupyter's file upload button.

This notebook will:
1. Check GPU and shared models
2. Load Qwen2.5-VL-7B directly
3. Load a sample document and check digital vs scanned
4. Run extraction
5. Inspect JSON output and experiment with formats
6. Process all pages
7. Launch Streamlit app

## 1. Check GPU and shared models

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total,memory.free --format=csv,noheader

In [ ]:
from pathlib import Path
import os

os.environ["HF_HOME"] = "/models/.cache/huggingface"
os.environ["HF_HUB_CACHE"] = "/models/.cache/huggingface"
os.environ["HF_HUB_OFFLINE"] = "1"

models_dir = Path("/models/.cache/huggingface")
if models_dir.exists():
    model_dirs = [d.name for d in models_dir.iterdir() if d.name.startswith("models--")]
    print(f"Shared models PVC mounted. {len(model_dirs)} model(s):")
    for m in sorted(model_dirs):
        print(f"  {m}")
    has_vlm = any("Qwen2.5-VL" in d for d in model_dirs)
    if has_vlm:
        print("\nQwen2.5-VL found.")
    else:
        print("\nWARNING: Qwen2.5-VL not found!")
        print("Run: python /models/provision_shared_models.py download Qwen/Qwen2.5-VL-7B-Instruct")
else:
    print("WARNING: /models/.cache/huggingface not found.")
    print("Is the shared-models data volume attached?")

## 2. Load the model

Load Qwen2.5-VL-7B directly with transformers. Takes ~1-2 min.

In [ ]:
import time
import torch
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

MODEL_NAME = "Qwen/Qwen2.5-VL-7B-Instruct"

print(f"Loading {MODEL_NAME}...")
t0 = time.time()

processor = AutoProcessor.from_pretrained(MODEL_NAME)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

elapsed = time.time() - t0
print(f"Model loaded in {elapsed:.1f}s")
print(f"Device: {model.device}")

In [ ]:
!pip install -q qwen-vl-utils

In [ ]:
from qwen_vl_utils import process_vision_info

def run_vlm(messages, max_tokens=4096):
    """Run inference on the loaded model."""
    text_input = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text_input], images=image_inputs, videos=video_inputs,
        padding=True, return_tensors="pt"
    ).to(model.device)
    with torch.no_grad():
        generated_ids = model.generate(**inputs, max_new_tokens=max_tokens)
    trimmed = [out[len(inp):] for inp, out in zip(inputs.input_ids, generated_ids)]
    return processor.batch_decode(trimmed, skip_special_tokens=True)[0]

def extract_text(text, prompt, max_tokens=4096):
    """Digital path: send extracted text to model."""
    full_prompt = f"{prompt}\n\n---\nDOCUMENT TEXT:\n---\n{text}"
    messages = [{"role": "user", "content": [{"type": "text", "text": full_prompt}]}]
    return run_vlm(messages, max_tokens)

def extract_image(image, prompt, max_tokens=4096):
    """Scanned path: send image to model."""
    import base64, io
    buf = io.BytesIO()
    image.save(buf, format="PNG")
    b64 = base64.b64encode(buf.getvalue()).decode()
    messages = [{"role": "user", "content": [
        {"type": "image", "image": f"data:image/png;base64,{b64}"},
        {"type": "text", "text": prompt},
    ]}]
    return run_vlm(messages, max_tokens)

print("Helper functions ready.")

In [ ]:
# ── Extraction schema prompt ──────────────────────────────────────────
EXTRACTION_PROMPT = """Analyze this document page and extract ALL information into the JSON structure below.

{
  "confidence_percentage": <float 0-100>,
  "confidence_narrative": "<why this confidence level>",
  "has_annotation": <true/false — handwritten marks or annotations>,
  "has_watermark": <true/false>,
  "signature_lines": {
    "has_signature_line": <true/false>,
    "has_valid_signature": <true/false>
  },
  "document_tags": ["<classification tag>", ...],
  "one_sentence_summary": "<one sentence describing this page content>",
  "document_details": {
    "application_id": "", "application_type": "", "title": "",
    "requested_amount": null, "completed_date": "", "sub_document_type": ""
  },
  "stakeholders": [{"name": "", "role": "", "organization": "", "email": "", "phone": ""}],
  "addresses": [{"type": "", "street": "", "city": "", "state": "", "zip": ""}],
  "tables": [
    {
      "classification": "Standard_Table",
      "rows": [{"<ColumnHeader>": "<value> [cite: N]"}]
    }
  ],
  "narrative_responses": [
    {
      "prompt_or_header": "<section heading>",
      "verbatim_text": "<exact text with [cite: N] after each statement>"
    }
  ]
}

RULES:
- Preserve ALL text VERBATIM — do not paraphrase, correct spelling, or summarize
- Number each logical element (heading, paragraph, table, list item) sequentially as [cite: N] starting from 1
- The page title/heading is [cite: 1]; subsequent paragraphs, tables, list items increment from there
- Tables: use column headers as keys; every cell value gets the table element's [cite: N]
- Preserve all dollar amounts, dates, percentages exactly as written
- If a field is not present, use empty string "", null, or empty array []
- Omit stakeholders/addresses arrays entirely if none exist on the page
- Output ONLY valid JSON, no markdown fences or extra text"""


# ── Filename metadata parser ─────────────────────────────────────────
def parse_filename(filename):
    """Parse structured filename into FileNameMetaData.

    Expected pattern:
      {Drawer}_AWD-{id}_{Field2}_{Field3}_{Field4}_{Field5}_{DocumentType}.{ext}
    """
    import re
    stem = Path(filename).stem
    ext = Path(filename).suffix.lstrip('.')

    awd_match = re.search(r'_AWD-', stem)
    if awd_match:
        drawer = stem[:awd_match.start()]
        rest = stem[awd_match.start() + 1:]  # drop leading _
        parts = rest.split('_')
        award_id = parts[0] if parts else ""
        field2 = parts[1] if len(parts) > 1 else ""
        field3 = parts[2] if len(parts) > 2 else ""
        field4 = parts[3] if len(parts) > 3 else ""
        field5 = parts[4] if len(parts) > 4 else ""
        doc_type = '_'.join(parts[5:]) if len(parts) > 5 else ""
    else:
        drawer = stem
        award_id = field2 = field3 = field4 = field5 = ""
        doc_type = stem

    doc_type_short = doc_type.rsplit('_', 1)[-1] if doc_type else ""

    return {
        "Drawer": drawer, "AwardID": award_id,
        "Field2": field2, "Field3": field3, "Field4": field4, "Field5": field5,
        "DocumentType": doc_type, "DocumentTypeShort": doc_type_short,
        "FileType": ext
    }


# ── Document-level assembly ──────────────────────────────────────────
def assemble_document_json(filename, page_results, model_name):
    """Combine per-page VLM results into the expected document-level JSON."""
    from datetime import datetime

    file_meta = parse_filename(filename)

    all_tables, all_narratives = [], []
    all_stakeholders, all_addresses = [], []
    all_tags = set()
    summaries = []
    has_annotation = has_watermark = False
    sig_info = {"PageNumber": None, "HasSignatureLine": False, "HasValidSignature": False}
    confidence_scores, confidence_narratives = [], []
    doc_details = {}

    for pr in page_results:
        pg = pr["page"]
        d = pr.get("extracted", {})

        for t in d.get("tables", []):
            all_tables.append({
                "PageNumber": pg,
                "TableClassification": t.get("classification", "Standard_Table"),
                "TableData": t.get("rows", [])
            })

        for n in d.get("narrative_responses", []):
            all_narratives.append({
                "SectionOrPage": f"PAGE {pg}",
                "PromptOrHeader": n.get("prompt_or_header", ""),
                "VerbatimText": n.get("verbatim_text", "")
            })

        for s in d.get("stakeholders", []):
            if any(v for v in s.values() if v):
                all_stakeholders.append(s)
        for a in d.get("addresses", []):
            if any(v for v in a.values() if v):
                all_addresses.append(a)

        all_tags.update(d.get("document_tags", []))
        if d.get("one_sentence_summary"):
            summaries.append(d["one_sentence_summary"])
        if d.get("has_annotation"): has_annotation = True
        if d.get("has_watermark"): has_watermark = True

        sig = d.get("signature_lines", {})
        if sig.get("has_signature_line"):
            sig_info = {"PageNumber": pg, "HasSignatureLine": True,
                        "HasValidSignature": sig.get("has_valid_signature", False)}

        if d.get("confidence_percentage") is not None:
            confidence_scores.append(d["confidence_percentage"])
        if d.get("confidence_narrative"):
            confidence_narratives.append(d["confidence_narrative"])

        if not doc_details and d.get("document_details"):
            doc_details = d["document_details"]

    avg_conf = round(sum(confidence_scores) / len(confidence_scores), 1) if confidence_scores else 0.0
    now = datetime.now().strftime("%Y-%m-%dT%H:%M:%S") + time.strftime("%Z")

    return [{
        "Filename": filename,
        "PageCount": len(page_results),
        "ConfidencePercentage": avg_conf,
        "ConfidenceNarrative": " ".join(confidence_narratives),
        "LLMModelAndVersion": model_name,
        "CurrentDateAndTime": now,
        "HasAnnotation": has_annotation,
        "HasWatermark": has_watermark,
        "SignatureLines": sig_info,
        "DocumentTags": sorted(all_tags),
        "OneSentenceNarrativeSummary": summaries,
        "FileNameMetaData": file_meta,
        "DocumentDetails": {
            "ApplicationID": doc_details.get("application_id", ""),
            "ApplicationType": doc_details.get("application_type", ""),
            "Title": doc_details.get("title", ""),
            "RequestedAmount": doc_details.get("requested_amount"),
            "CompletedDate": doc_details.get("completed_date", ""),
            "SubDocumentType": doc_details.get("sub_document_type", "")
        },
        "Stakeholders": all_stakeholders,
        "AddressesCollection": all_addresses,
        "TablesCollection": all_tables,
        "NarrativeResponses": all_narratives,
        "OtherMetadata": {}
    }]


print("Extraction prompt, filename parser, and assembly function ready.")

## 3. Load a sample document

Upload your sample PDFs/TIFFs to `/ocr/` using Jupyter's file upload button.

In [ ]:
ocr_dir = Path("/ocr")
files = [f for f in ocr_dir.iterdir() if f.is_file()]
print("Files in /ocr/:")
for f in sorted(files):
    print(f"  {f.name} ({f.stat().st_size / 1024:.0f} KB)")
if not files:
    print("\nNo files found. Upload your sample docs to /ocr/ first.")

In [ ]:
DOC_PATH = Path("/ocr/sample.pdf")  # <-- UPDATE THIS

assert DOC_PATH.exists(), f"File not found: {DOC_PATH}"
print(f"Document: {DOC_PATH.name} ({DOC_PATH.stat().st_size / 1024:.0f} KB)")

## 4. Check digital vs scanned

In [ ]:
import fitz

doc = fitz.open(str(DOC_PATH))
print(f"Pages: {len(doc)}\n")

page_info = []
for i, page in enumerate(doc):
    text = page.get_text("text").strip()
    has_text = len(text) >= 50
    page_info.append({"page": i, "text": text, "has_text": has_text})
    status = "DIGITAL" if has_text else "SCANNED"
    print(f"Page {i+1}: {status} ({len(text)} chars)")
    if has_text:
        print(f"  Preview: {text[:150]}...")
    print()

digital = sum(1 for p in page_info if p["has_text"])
scanned = sum(1 for p in page_info if not p["has_text"])
print(f"Summary: {digital} digital, {scanned} scanned")
doc.close()

from PIL import Image

PAGE_IDX = 0  # Change this to test different pages
info = page_info[PAGE_IDX]

t0 = time.time()

if info["has_text"]:
    print(f"Page {PAGE_IDX+1}: Using DIGITAL path\n")
    raw_result = extract_text(info["text"], EXTRACTION_PROMPT)
else:
    print(f"Page {PAGE_IDX+1}: Using SCANNED path\n")
    doc = fitz.open(str(DOC_PATH))
    mat = fitz.Matrix(2.0, 2.0)
    pix = doc[PAGE_IDX].get_pixmap(matrix=mat)
    img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
    doc.close()
    print(f"Rendered: {img.width}x{img.height}")
    display(img.resize((img.width // 3, img.height // 3)))
    raw_result = extract_image(img, EXTRACTION_PROMPT)

elapsed = time.time() - t0
print(f"Extraction took {elapsed:.1f}s")

In [ ]:
from PIL import Image

PAGE_IDX = 0  # Change this to test different pages
info = page_info[PAGE_IDX]

# Prompt — try different ones from step 7!
PROMPT = """Extract all information from this document.

Return a JSON object with these fields (omit any not present):
  "document_type", "award_number", "sponsor", "pi", "institution",
  "project_title", "award_amount", "project_start", "project_end",
  "fa_rate", "additional_fields"

Preserve ALL dollar amounts and dates exactly. Output only valid JSON."""

t0 = time.time()

if info["has_text"]:
    print(f"Page {PAGE_IDX+1}: Using DIGITAL path\n")
    result = extract_text(info["text"], PROMPT)
else:
    print(f"Page {PAGE_IDX+1}: Using SCANNED path\n")
    doc = fitz.open(str(DOC_PATH))
    mat = fitz.Matrix(2.0, 2.0)
    pix = doc[PAGE_IDX].get_pixmap(matrix=mat)
    img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
    doc.close()
    print(f"Rendered: {img.width}x{img.height}")
    display(img.resize((img.width // 3, img.height // 3)))
    result = extract_image(img, PROMPT)

elapsed = time.time() - t0
print(f"\nExtraction took {elapsed:.1f}s")

import json

# Strip markdown code fences if present
cleaned = raw_result.strip()
if cleaned.startswith("```"):
    cleaned = cleaned.split("\n", 1)[1]
    cleaned = cleaned.rsplit("```", 1)[0]

try:
    parsed = json.loads(cleaned)
    print("Valid JSON from VLM\n")
    print(json.dumps(parsed, indent=2))
except json.JSONDecodeError as e:
    print(f"Invalid JSON: {e}\n")
    print("Raw output:")
    print(raw_result)

## 7. Extraction prompt reference

The `EXTRACTION_PROMPT` above produces structured JSON with:
- **Document tags** and one-sentence summary
- **Confidence** percentage and narrative
- **Annotation / watermark / signature** detection
- **Tables** with column headers as keys and `[cite: N]` markers
- **Narrative responses** with verbatim text and citations
- **Stakeholders** and **addresses** (when present)
- **Document details** (title, application ID, amounts, dates)

Edit `EXTRACTION_PROMPT` to add or remove fields as needed.

# Quick test: parse the current document filename
meta = parse_filename(DOC_PATH.name)
print("FileNameMetaData:")
for k, v in meta.items():
    print(f"  {k}: {v!r}")

In [ ]:
PROMPT_KV = """Extract all labeled data points from this document as key-value pairs.
Return a JSON object where keys are the field names and values are their values.
Preserve ALL values exactly. Output only valid JSON."""

PROMPT_BUDGET = """Extract budget information from this document.
Return a JSON object with: award_number, budget_period,
categories (array of {category, items: [{description, amount}], subtotal}),
total_direct, fa_rate, fa_base, total_indirect, total, cost_sharing, notes.
Preserve ALL dollar amounts exactly. Output only valid JSON."""

PROMPT_TERMS = """Extract terms and conditions from this document.
Return a JSON object with: document_title, effective_date,
sections (array of {number, title, text, subsections}),
definitions, references.
Preserve exact wording. Output only valid JSON."""

PROMPT_TEXT = """Extract all text from this document exactly as it appears.
Preserve the original reading order, line breaks, and structure.
Output only the extracted text."""

print("Copy one of these into PROMPT in step 5 and re-run.")
print("Available: PROMPT_KV, PROMPT_BUDGET, PROMPT_TERMS, PROMPT_TEXT")

import json
from PIL import Image

results = []
doc = fitz.open(str(DOC_PATH))

for info in page_info:
    page_num = info["page"] + 1
    t0 = time.time()

    if info["has_text"]:
        raw = extract_text(info["text"], EXTRACTION_PROMPT)
        method = "text_extraction"
    else:
        mat = fitz.Matrix(2.0, 2.0)
        pix = doc[info["page"]].get_pixmap(matrix=mat)
        img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
        raw = extract_image(img, EXTRACTION_PROMPT)
        method = "vlm_ocr"

    elapsed = time.time() - t0

    # Parse VLM JSON output
    cleaned = raw.strip()
    if cleaned.startswith("```"):
        cleaned = cleaned.split("\n", 1)[1]
        cleaned = cleaned.rsplit("```", 1)[0]
    try:
        extracted = json.loads(cleaned)
    except json.JSONDecodeError:
        print(f"  WARNING: Page {page_num} returned invalid JSON, storing raw text")
        extracted = {"raw_text": raw, "confidence_percentage": 0,
                     "confidence_narrative": "Failed to parse structured output"}

    results.append({"page": page_num, "method": method,
                     "elapsed_ms": round(elapsed * 1000, 1), "extracted": extracted})
    print(f"Page {page_num}: {method} ({elapsed:.1f}s)")

doc.close()
print(f"\nDone. {len(results)} pages processed.")

In [ ]:
output = assemble_document_json(
    filename=DOC_PATH.name,
    page_results=results,
    model_name=MODEL_NAME
)

out_path = Path(f"/ocr/{DOC_PATH.stem}_extracted.json")
out_path.write_text(json.dumps(output, indent=2))
print(f"Saved to {out_path}\n")
print(json.dumps(output, indent=2))

In [ ]:
output = {
    "source_file": str(DOC_PATH),
    "total_pages": len(results),
    "digital_pages": sum(1 for r in results if r["method"] == "text_extraction"),
    "scanned_pages": sum(1 for r in results if r["method"] == "vlm_ocr"),
    "pages": results,
}
out_path = Path(f"/ocr/{DOC_PATH.stem}_extracted.json")
out_path.write_text(json.dumps(output, indent=2))
print(f"Saved to {out_path}")

## 9. Test Streamlit app

Launches the extraction server and Streamlit UI.
Access at: `https://<cluster-host>/<project>/ocr-setup/proxy/8501/`

**Requires a Custom URL tool on port 8501** in the workspace config.

In [ ]:
# TODO: Streamlit app currently requires a vLLM/Ollama endpoint
# (ocr_server.py talks to an OpenAI-compatible API, not the local model).
# To test Streamlit, start vLLM in a terminal first:
#
#   python -m vllm.entrypoints.openai.api_server \
#       --model Qwen/Qwen2.5-VL-7B-Instruct --dtype auto \
#       --max-model-len 8192 --limit-mm-per-prompt image=1
#
# Then run in another terminal:
#
#   cd /tmp/KohakuRAG_UI
#   LLM_BASE_URL=http://localhost:8000/v1 python ocr_app/scripts/ocr_server.py &
#   OCR_SERVICE_URL=http://localhost:8090 streamlit run ocr_app/app.py \
#       --server.port=8501 --server.address=0.0.0.0 --server.headless=true

print("See instructions above to test Streamlit.")
print("The notebook extraction (steps 2-8) uses the model directly — no vLLM needed.")
print("Streamlit requires vLLM because the extraction server talks to an API endpoint.")